# DAVID-AI code model: train nanoGPT on a free Colab T4

**First:** Runtime → Change runtime type → **T4 GPU** → Save. Then run the cells top to bottom.

Your checkpoints are saved to Google Drive, so if Colab disconnects you lose nothing: just run the cells again and use the **Resume** cell.

In [ ]:
# 1. Check the GPU (should say Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get nanoGPT + the DAVID-AI scripts. PyTorch is already installed on Colab (don't reinstall it).
%cd /content
!rm -rf nanoGPT DAVID-AI
!git clone -q https://github.com/karpathy/nanoGPT.git
!git clone -q https://github.com/seundeeun-collab/DAVID-AI.git
!pip install -q tiktoken datasets transformers safetensors onnx onnxruntime
%cd /content/nanoGPT
!mkdir -p data/code
!cp ../DAVID-AI/nanogpt/prepare_code.py data/code/prepare.py
!cp ../DAVID-AI/nanogpt/train_code.py config/train_code.py
!cp ../DAVID-AI/nanogpt/export.py ../DAVID-AI/nanogpt/generate.py ../DAVID-AI/nanogpt/train_with_dashboard.py .
import torch; print('PyTorch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime to T4 GPU')

## Optional warm-up: Shakespeare (about 10-15 minutes)
Proves the GPU training works before the big run.

In [ ]:
!python data/shakespeare_char/prepare.py
!python train.py config/train_shakespeare_char.py --max_iters=2000 --lr_decay_iters=2000 --dtype=float16 --compile=False
!python sample.py --out_dir=out-shakespeare-char --num_samples=1 --max_new_tokens=300

## Real training: code in 11 languages

In [ ]:
# 3. Save checkpoints to Google Drive (survives disconnects)
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/david-ai/out-code'
!mkdir -p {OUT}

In [ ]:
# 4. Download + tokenize real open-source code (Python, JavaScript, TypeScript, HTML, CSS, Java, C++, Go, Rust, PHP, SQL)
# Bigger --per_lang = more data = better model, but slower. Stops after --max_minutes either way.
!python data/code/prepare.py --per_lang 20000 --max_minutes 60

In [ ]:
# 5. Connect the live dashboard (optional). In Colab's left bar click the key icon (Secrets) and add
#    SITE_URL      = your website address, e.g. https://xxxx.lovable.app
#    INGEST_TOKEN  = the ingest token from your website's Settings page
import os
from google.colab import userdata
for k in ('SITE_URL', 'INGEST_TOKEN'):
    try: os.environ[k] = userdata.get(k)
    except Exception: print(f'{k} not set: training will run without the live dashboard')

In [ ]:
# 6. TRAIN (first time)
!python train_with_dashboard.py config/train_code.py --out_dir={OUT}

In [ ]:
# 6b. RESUME after a disconnect (run cells 2, 3, 4, 5 again first, then this instead of 6)
!python train_with_dashboard.py config/train_code.py --out_dir={OUT} --init_from=resume

In [ ]:
# 7. Test it
!python generate.py --out_dir={OUT} --lang python --task "Write a function to add two numbers"
!python generate.py --out_dir={OUT} --lang javascript --task "Write a function to add two numbers"

In [ ]:
# 8. Export files for the website (model.safetensors, config.json, tokenizer.json, model.onnx, model.int8.onnx)
!python export.py --out_dir={OUT} --dest={OUT}/export
!cd {OUT}/export && zip -q -r /content/david-ai-code-model.zip . && ls -lh
from google.colab import files
files.download('/content/david-ai-code-model.zip')